# Baby Step 2 — Test Generalization Across Five Litigation Contexts

**Project:** Corporate Civil Litigation Exo-Brain  
**Author:** Alejandro Reynoso  
**Persistent vault:** `/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain`

## Objective

Baby Step 2 tests whether the first strategy-development loop genuinely generalizes across five materially different corporate civil disputes.

Baby Step 1 applied one common retrieval and strategy-scoring architecture to all five active matters. That produced five Recommendation V1 records. Baby Step 2 now asks:

> Does one common reasoning model behave sensibly across injunctions, accounting disputes, licensing conflicts, governance deadlock, and supply-chain litigation?

This notebook does not create Recommendation V2. It creates a **generalization report**, identifies structural bias, proposes matter-specific reasoning profiles, and records which Recommendation V1 records should remain stable, be qualified, or be reopened.


## Why generalization must be tested

A model that performs well on one matter may fail on another because different disputes depend on different legal and operational dimensions.

Examples:

- injunction matters depend heavily on irreparable harm and procedural urgency;
- purchase-price disputes depend more on contract text and accounting methodology;
- technology disputes depend on technical evidence and confidentiality;
- joint-venture deadlock depends on governance mechanisms and remedy design;
- supply disputes depend on causation, mitigation, and damages limitations.

A single global weighting scheme may therefore create hidden bias.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import csv
import datetime
import statistics
import math
from collections import defaultdict, Counter

VAULT = Path(r"/content/drive/MyDrive/Alejandro-Reynoso-Corporate-Civil-Litigation-ExoBrain")

if not VAULT.exists():
    raise FileNotFoundError("Run Baby Step 0 first.")

state_path = VAULT / "00_System" / "Workflow_State.json"
state = json.loads(state_path.read_text(encoding="utf-8"))

if 1 not in state.get("completed_steps", []):
    raise RuntimeError("Baby Step 1 is not complete.")

matters = json.loads(
    (VAULT / "data" / "active_matters.json").read_text(encoding="utf-8")
)

recommendations_v1 = json.loads(
    (VAULT / "data" / "baby_step_1_recommendations_v1.json").read_text(encoding="utf-8")
)

evaluations_v1 = json.loads(
    (VAULT / "data" / "baby_step_1_strategy_evaluations.json").read_text(encoding="utf-8")
)

authority_sets = json.loads(
    (VAULT / "data" / "baby_step_1_authority_sets.json").read_text(encoding="utf-8")
)

print("Active matters:", len(matters))
print("Recommendation V1 records:", len(recommendations_v1))


## Common-model baseline

The Baby Step 1 model used the same dimensions for every matter:

- legal support;
- evidence readiness;
- procedural fit;
- risk alignment;
- institutional objective.

Those dimensions are reasonable, but their relative importance may differ by dispute type.

Baby Step 2 preserves the original model as the baseline and introduces matter-specific profiles without overwriting the prior result.


In [ ]:
COMMON_WEIGHTS = {
    "legal_support": 0.30,
    "evidence_readiness": 0.20,
    "procedural_fit": 0.20,
    "risk_alignment": 0.15,
    "objective_alignment": 0.15
}

MATTER_PROFILES = {
    "Breach of Shareholders Agreement": {
        "legal_support": 0.25,
        "evidence_readiness": 0.15,
        "procedural_fit": 0.30,
        "risk_alignment": 0.15,
        "objective_alignment": 0.15,
        "special_dimension": "irreparable_harm_and_control_preservation"
    },
    "Post-M&A Purchase Price Dispute": {
        "legal_support": 0.25,
        "evidence_readiness": 0.30,
        "procedural_fit": 0.15,
        "risk_alignment": 0.10,
        "objective_alignment": 0.20,
        "special_dimension": "contract_text_and_accounting_methodology"
    },
    "Technology Licensing Dispute": {
        "legal_support": 0.20,
        "evidence_readiness": 0.30,
        "procedural_fit": 0.20,
        "risk_alignment": 0.10,
        "objective_alignment": 0.20,
        "special_dimension": "technical_proof_and_confidentiality"
    },
    "Joint Venture Deadlock": {
        "legal_support": 0.20,
        "evidence_readiness": 0.15,
        "procedural_fit": 0.20,
        "risk_alignment": 0.15,
        "objective_alignment": 0.30,
        "special_dimension": "governance_functionality_and_value_preservation"
    },
    "Supply Agreement Dispute": {
        "legal_support": 0.25,
        "evidence_readiness": 0.25,
        "procedural_fit": 0.15,
        "risk_alignment": 0.15,
        "objective_alignment": 0.20,
        "special_dimension": "causation_mitigation_and_damages_limits"
    }
}

for cause, weights in MATTER_PROFILES.items():
    numeric = {k:v for k,v in weights.items() if k != "special_dimension"}
    assert abs(sum(numeric.values()) - 1.0) < 1e-9, cause

(VAULT / "00_System" / "Baby_Step_2_Matter_Profiles.json").write_text(
    json.dumps({
        "common_weights": COMMON_WEIGHTS,
        "matter_profiles": MATTER_PROFILES
    }, indent=2),
    encoding="utf-8"
)

print("Matter-specific profiles created:", len(MATTER_PROFILES))


## Generalization test design

For every strategy alternative, Baby Step 2 recalculates the score using the matter-specific profile.

The test measures:

- whether the preferred strategy changes;
- how much the score changes;
- whether the confidence ranking changes;
- whether the common model systematically overweights or underweights one dimension;
- whether Recommendation V1 should remain stable, be qualified, or be reopened.

No recommendation is overwritten.


In [ ]:
def rescore(option, profile):
    weights = {
        k:v for k,v in profile.items()
        if k != "special_dimension"
    }

    return round(
        option["legal_support"] * weights["legal_support"]
        + option["evidence_readiness"] * weights["evidence_readiness"]
        + option["procedural_fit"] * weights["procedural_fit"]
        + option["risk_alignment"] * weights["risk_alignment"]
        + option["objective_alignment"] * weights["objective_alignment"],
        2
    )

generalization_results = {}

for matter in matters:
    mid = matter["matter_id"]
    cause = matter["cause_of_action"]
    profile = MATTER_PROFILES[cause]

    rescored = []

    for option in evaluations_v1[mid]:
        updated = dict(option)
        updated["common_model_score"] = option["strategy_score"]
        updated["profiled_score"] = rescore(option, profile)
        updated["score_delta"] = round(
            updated["profiled_score"] - updated["common_model_score"],
            2
        )
        rescored.append(updated)

    rescored.sort(
        key=lambda x: -x["profiled_score"]
    )

    v1 = next(
        r for r in recommendations_v1
        if r["matter_id"] == mid
    )

    preferred_changed = (
        rescored[0]["strategy"] != v1["preferred_strategy"]
    )

    max_abs_delta = max(
        abs(item["score_delta"])
        for item in rescored
    )

    if preferred_changed:
        status = "REOPEN"
    elif max_abs_delta >= 5:
        status = "QUALIFY"
    else:
        status = "STABLE"

    generalization_results[mid] = {
        "matter_id": mid,
        "cause_of_action": cause,
        "special_dimension": profile["special_dimension"],
        "recommendation_v1": v1["preferred_strategy"],
        "profiled_preferred_strategy": rescored[0]["strategy"],
        "preferred_changed": preferred_changed,
        "max_absolute_score_delta": round(max_abs_delta, 2),
        "status": status,
        "rescored_alternatives": rescored
    }

(VAULT / "data" / "baby_step_2_generalization_results.json").write_text(
    json.dumps(generalization_results, indent=2),
    encoding="utf-8"
)

print(json.dumps(generalization_results, indent=2))


## Structural-bias analysis

Baby Step 2 also tests whether the common model systematically privileges a particular strategy type.

The relevant question is not merely whether a recommendation changes. It is whether the architecture tends to favor aggressive, procedural, negotiated, evidentiary, or governance-oriented strategies regardless of matter type.


In [ ]:
bias_records = []

for matter in matters:
    mid = matter["matter_id"]
    result = generalization_results[mid]

    original_order = [
        item["strategy"]
        for item in sorted(
            evaluations_v1[mid],
            key=lambda x: -x["strategy_score"]
        )
    ]

    profiled_order = [
        item["strategy"]
        for item in result["rescored_alternatives"]
    ]

    movement = {
        strategy: original_order.index(strategy) - profiled_order.index(strategy)
        for strategy in original_order
    }

    bias_records.append({
        "matter_id": mid,
        "cause_of_action": matter["cause_of_action"],
        "original_order": original_order,
        "profiled_order": profiled_order,
        "rank_movement": movement,
        "status": result["status"]
    })

(VAULT / "data" / "baby_step_2_bias_analysis.json").write_text(
    json.dumps(bias_records, indent=2),
    encoding="utf-8"
)

status_counts = Counter(
    record["status"]
    for record in bias_records
)

print("Generalization status counts:", dict(status_counts))


## Sensitivity analysis

The model is then tested under controlled perturbations.

For each matter-specific profile, the notebook increases and decreases each weight by 20%, renormalizes the remaining weights, and observes whether the preferred strategy changes.

A strategy that changes under small perturbations is fragile and should receive lower confidence.


In [ ]:
def perturb_profile(profile, target, factor):
    numeric = {
        k:v for k,v in profile.items()
        if k != "special_dimension"
    }

    numeric[target] *= factor
    total = sum(numeric.values())

    return {
        k:v/total
        for k,v in numeric.items()
    }

def rescore_with_weights(option, weights):
    return round(
        option["legal_support"] * weights["legal_support"]
        + option["evidence_readiness"] * weights["evidence_readiness"]
        + option["procedural_fit"] * weights["procedural_fit"]
        + option["risk_alignment"] * weights["risk_alignment"]
        + option["objective_alignment"] * weights["objective_alignment"],
        2
    )

sensitivity_results = {}

for matter in matters:
    mid = matter["matter_id"]
    profile = MATTER_PROFILES[matter["cause_of_action"]]
    base_preferred = generalization_results[mid]["profiled_preferred_strategy"]

    tests = []

    for dimension in COMMON_WEIGHTS:
        for factor in [0.80, 1.20]:
            weights = perturb_profile(profile, dimension, factor)

            ranked = sorted(
                [
                    {
                        "strategy": option["strategy"],
                        "score": rescore_with_weights(option, weights)
                    }
                    for option in evaluations_v1[mid]
                ],
                key=lambda x: -x["score"]
            )

            tests.append({
                "dimension": dimension,
                "factor": factor,
                "preferred_strategy": ranked[0]["strategy"],
                "preferred_changed": ranked[0]["strategy"] != base_preferred,
                "top_score": ranked[0]["score"]
            })

    switches = sum(
        1 for test in tests
        if test["preferred_changed"]
    )

    fragility = round(
        100 * switches / len(tests),
        2
    )

    sensitivity_results[mid] = {
        "matter_id": mid,
        "base_profiled_preferred": base_preferred,
        "tests": tests,
        "switch_count": switches,
        "fragility_score": fragility
    }

(VAULT / "data" / "baby_step_2_sensitivity_results.json").write_text(
    json.dumps(sensitivity_results, indent=2),
    encoding="utf-8"
)

for mid, result in sensitivity_results.items():
    print(mid, "fragility:", result["fragility_score"])


## Recommendation review state

Baby Step 2 does not issue Recommendation V2.

Instead, it creates a governed review state:

- **STABLE** — Recommendation V1 remains the working baseline;
- **QUALIFY** — Recommendation V1 remains, but its rationale or confidence must be narrowed;
- **REOPEN** — Recommendation V1 should be reconsidered before committee use.

This preserves temporal integrity.


In [ ]:
review_records = []

for matter in matters:
    mid = matter["matter_id"]
    result = generalization_results[mid]
    sensitivity = sensitivity_results[mid]
    v1 = next(
        r for r in recommendations_v1
        if r["matter_id"] == mid
    )

    original_confidence = v1["confidence_score"]
    fragility_penalty = sensitivity["fragility_score"] * 0.20

    if result["status"] == "STABLE":
        status_penalty = 0
    elif result["status"] == "QUALIFY":
        status_penalty = 7
    else:
        status_penalty = 15

    adjusted_confidence = round(
        max(
            0,
            original_confidence
            - fragility_penalty
            - status_penalty
        ),
        2
    )

    record = {
        "review_id": f"REV-{mid}-BS2",
        "matter_id": mid,
        "recommendation_v1_id": v1["recommendation_id"],
        "generalization_status": result["status"],
        "original_strategy": v1["preferred_strategy"],
        "profiled_preferred_strategy": result["profiled_preferred_strategy"],
        "original_confidence": original_confidence,
        "adjusted_confidence_for_review": adjusted_confidence,
        "fragility_score": sensitivity["fragility_score"],
        "special_dimension": result["special_dimension"],
        "recommendation_v2_created": False,
        "human_review_required": True,
        "synthetic": True
    }

    review_records.append(record)

(VAULT / "data" / "baby_step_2_review_records.json").write_text(
    json.dumps(review_records, indent=2),
    encoding="utf-8"
)

print(json.dumps(review_records, indent=2))


In [ ]:
def write_note(path, lines):
    path.write_text(
        "\n".join(lines).strip() + "\n",
        encoding="utf-8"
    )

review_dir = VAULT / "08_Recommendations" / "Generalization_Reviews"
review_dir.mkdir(parents=True, exist_ok=True)

for record in review_records:
    result = generalization_results[record["matter_id"]]

    lines = [
        "---",
        f"review_id: {record['review_id']}",
        f"matter_id: {record['matter_id']}",
        "baby_step: 2",
        f"generalization_status: {record['generalization_status']}",
        "recommendation_v2_created: false",
        "human_review_required: true",
        "synthetic: true",
        "---",
        "",
        f"# {record['review_id']} — Generalization Review",
        "",
        "## Recommendation under review",
        "",
        f"[[../{record['recommendation_v1_id']}]]",
        "",
        "## Original strategy",
        "",
        record["original_strategy"],
        "",
        "## Profiled preferred strategy",
        "",
        record["profiled_preferred_strategy"],
        "",
        "## Matter-specific dimension",
        "",
        record["special_dimension"],
        "",
        "## Generalization status",
        "",
        f"**{record['generalization_status']}**",
        "",
        "## Confidence",
        "",
        f"- Original confidence: {record['original_confidence']}/100",
        f"- Adjusted review confidence: {record['adjusted_confidence_for_review']}/100",
        f"- Fragility score: {record['fragility_score']}/100",
        "",
        "## Alternative ranking under matter-specific profile",
        ""
    ]

    for idx, option in enumerate(
        result["rescored_alternatives"],
        start=1
    ):
        lines.append(
            f"{idx}. {option['strategy']} — "
            f"profiled score {option['profiled_score']} "
            f"(delta {option['score_delta']:+.2f})"
        )

    lines += [
        "",
        "## Governance",
        "",
        "No Recommendation V2 is created in Baby Step 2.",
        "Human review is required before any recommendation is revised."
    ]

    write_note(
        review_dir / f"{record['review_id']}.md",
        lines
    )

print("Generalization review notes:", len(list(review_dir.glob("*.md"))))


## Comparative operating report

The report compares the five matters and identifies:

- which recommendations are stable;
- which require qualification;
- which should be reopened;
- which matters are most sensitive to assumptions;
- where the common model was structurally incomplete.


In [ ]:
report = [
    "# Baby Step 2 — Generalization and Structural-Bias Report",
    "",
    "## Executive conclusion",
    "",
    "The common operating loop was tested across five heterogeneous civil-litigation contexts.",
    "Matter-specific profiles reveal whether Recommendation V1 is stable, should be qualified, or should be reopened.",
    "",
    "## Portfolio summary",
    ""
]

for status in ["STABLE", "QUALIFY", "REOPEN"]:
    report.append(
        f"- {status}: {status_counts.get(status,0)} matters"
    )

report += [
    "",
    "## Matter-by-matter results",
    ""
]

for record in review_records:
    matter = next(
        m for m in matters
        if m["matter_id"] == record["matter_id"]
    )

    report += [
        f"### {record['matter_id']} — {matter['caption']}",
        "",
        f"- Cause of action: {matter['cause_of_action']}",
        f"- Matter-specific dimension: {record['special_dimension']}",
        f"- Recommendation V1: {record['original_strategy']}",
        f"- Profiled preferred strategy: {record['profiled_preferred_strategy']}",
        f"- Status: **{record['generalization_status']}**",
        f"- Fragility: {record['fragility_score']}/100",
        f"- Adjusted review confidence: {record['adjusted_confidence_for_review']}/100",
        f"- Review note: [[../08_Recommendations/Generalization_Reviews/{record['review_id']}]]",
        ""
    ]

report += [
    "## Requested decision",
    "",
    "Accept the matter-specific profiles as the generalization-testing baseline.",
    "Do not create Recommendation V2 yet.",
    "",
    "## Not authorized",
    "",
    "- No filing or service",
    "- No party or court contact",
    "- No external counsel instruction",
    "- No settlement offer",
    "- No external legal advice",
    "- No deletion or overwrite of Recommendation V1"
]

write_note(
    VAULT / "10_Reports" / "Baby_Step_2_Generalization_Report.md",
    report
)


## Human decision — DEC-002

DEC-002 accepts the generalization test and matter-specific reasoning profiles as an internal model-risk baseline.

It authorizes:

- deeper analysis of qualified and reopened matters;
- preservation of Recommendation V1;
- matter-specific confidence adjustment;
- preparation for provenance and contradiction controls.

It does not authorize Recommendation V2.


In [ ]:
DECISION = {
    "decision_id": "DEC-002",
    "date": datetime.date.today().isoformat(),
    "title": "Accept Generalization Test and Matter-Specific Profiles",
    "decision": (
        "Accept the Baby Step 2 generalization results as the internal "
        "model-risk baseline while preserving all Recommendation V1 records."
    ),
    "stable_matters": [
        r["matter_id"]
        for r in review_records
        if r["generalization_status"] == "STABLE"
    ],
    "qualified_matters": [
        r["matter_id"]
        for r in review_records
        if r["generalization_status"] == "QUALIFY"
    ],
    "reopened_matters": [
        r["matter_id"]
        for r in review_records
        if r["generalization_status"] == "REOPEN"
    ],
    "permitted_next_actions": [
        "deeper internal review of qualified and reopened matters",
        "preserve Recommendation V1",
        "apply matter-specific reasoning profiles",
        "design provenance and contradiction controls"
    ],
    "not_authorized": [
        "Recommendation V2",
        "filing",
        "service",
        "party contact",
        "court contact",
        "external counsel instruction",
        "settlement offer",
        "external legal advice",
        "external distribution"
    ],
    "synthetic": True
}

(VAULT / "09_Decisions" / "DEC-002.json").write_text(
    json.dumps(DECISION, indent=2),
    encoding="utf-8"
)

decision_lines = [
    "# DEC-002 — Accept Generalization Test and Matter-Specific Profiles",
    "",
    f"**Date:** {DECISION['date']}",
    "",
    "## Decision",
    "",
    DECISION["decision"],
    "",
    "## Stable matters",
    ""
]

decision_lines += [
    f"- [[../02_Active_Matters/{mid}]]"
    for mid in DECISION["stable_matters"]
] or ["- None"]

decision_lines += [
    "",
    "## Qualified matters",
    ""
]

decision_lines += [
    f"- [[../02_Active_Matters/{mid}]]"
    for mid in DECISION["qualified_matters"]
] or ["- None"]

decision_lines += [
    "",
    "## Reopened matters",
    ""
]

decision_lines += [
    f"- [[../02_Active_Matters/{mid}]]"
    for mid in DECISION["reopened_matters"]
] or ["- None"]

decision_lines += [
    "",
    "## Permitted next actions",
    ""
]

decision_lines += [
    f"- {item}"
    for item in DECISION["permitted_next_actions"]
]

decision_lines += [
    "",
    "## Not authorized",
    ""
]

decision_lines += [
    f"- {item}"
    for item in DECISION["not_authorized"]
]

write_note(
    VAULT / "09_Decisions" / "DEC-002.md",
    decision_lines
)


## Hot-cache refresh

The hot cache now records:

- Recommendation V1 remains authoritative as the current internal baseline;
- each matter has a generalization status;
- no Recommendation V2 exists;
- the next problem is provenance, claim-level support, and contradiction control.


In [ ]:
hot_cache = [
    "# Current State — Hot Cache",
    "",
    "## Current universe",
    "",
    "- 1,000 synthetic precedents",
    "- 5 active synthetic corporate civil matters",
    "",
    "## Current recommendation state",
    "",
    "- Recommendation Version 1 remains the current internal baseline.",
    "- No Recommendation Version 2 exists.",
    "",
    "## Generalization status",
    ""
]

for record in review_records:
    hot_cache.append(
        f"- {record['matter_id']}: "
        f"**{record['generalization_status']}** — "
        f"[[../08_Recommendations/Generalization_Reviews/{record['review_id']}]]"
    )

hot_cache += [
    "",
    "## Current decision",
    "",
    "- [[../09_Decisions/DEC-002]]",
    "",
    "## Permitted",
    "",
    "- Internal model-risk review",
    "- Matter-specific profile analysis",
    "- Deeper review of qualified and reopened matters",
    "- Design provenance and contradiction controls",
    "",
    "## Prohibited",
    "",
    "- Recommendation V2",
    "- Filing or service",
    "- Party or court contact",
    "- Settlement offers",
    "- External legal advice",
    "- Overwriting Recommendation V1",
    "",
    "## Next permitted experiment",
    "",
    "Add provenance, atomic claims, authority treatment, and contradiction control."
]

write_note(
    VAULT / "12_Hot_Cache" / "Current_State.md",
    hot_cache
)


## Validation

Baby Step 2 passes only if:

- five generalization reviews exist;
- all five Recommendation V1 records remain unchanged;
- zero Recommendation V2 records exist;
- every matter receives STABLE, QUALIFY, or REOPEN status;
- sensitivity testing exists for every matter;
- DEC-002 exists;
- the comparative report and audit files exist.


In [ ]:
errors = []

review_notes = list(
    (VAULT / "08_Recommendations" / "Generalization_Reviews").glob("*.md")
)

v1_notes = list(
    (VAULT / "08_Recommendations").glob("REC-*-V001.md")
)

v2_notes = list(
    (VAULT / "08_Recommendations").glob("REC-*-V002.md")
)

if len(review_notes) != 5:
    errors.append(
        f"Expected 5 review notes, found {len(review_notes)}"
    )

if len(v1_notes) != 5:
    errors.append(
        f"Expected 5 Recommendation V1 notes, found {len(v1_notes)}"
    )

if v2_notes:
    errors.append(
        "Recommendation V2 exists prematurely"
    )

valid_statuses = {"STABLE", "QUALIFY", "REOPEN"}

for record in review_records:
    if record["generalization_status"] not in valid_statuses:
        errors.append(
            f"{record['matter_id']}: invalid status"
        )

    if record["matter_id"] not in sensitivity_results:
        errors.append(
            f"{record['matter_id']}: missing sensitivity result"
        )

required_files = [
    VAULT / "09_Decisions" / "DEC-002.md",
    VAULT / "09_Decisions" / "DEC-002.json",
    VAULT / "10_Reports" / "Baby_Step_2_Generalization_Report.md",
    VAULT / "12_Hot_Cache" / "Current_State.md",
    VAULT / "data" / "baby_step_2_generalization_results.json",
    VAULT / "data" / "baby_step_2_bias_analysis.json",
    VAULT / "data" / "baby_step_2_sensitivity_results.json",
    VAULT / "data" / "baby_step_2_review_records.json"
]

for path in required_files:
    if not path.exists():
        errors.append(
            f"Missing required output: {path}"
        )

validation = {
    "validated_at": datetime.datetime.now().isoformat(),
    "review_note_count": len(review_notes),
    "recommendation_v1_count": len(v1_notes),
    "recommendation_v2_count": len(v2_notes),
    "status_counts": dict(status_counts),
    "decision": "DEC-002",
    "errors": errors,
    "passed": len(errors) == 0
}

(VAULT / "11_Audit" / "Baby_Step_2_Validation.json").write_text(
    json.dumps(validation, indent=2),
    encoding="utf-8"
)

assert validation["passed"], errors

print(json.dumps(validation, indent=2))
print("BABY STEP 2 PASSED")


In [ ]:
state.update({
    "completed_steps": sorted(
        set(state.get("completed_steps", []) + [2])
    ),
    "current_step": 2,
    "next_step": 3,
    "decision": "DEC-002",
    "recommendation_count": 5,
    "current_recommendation_version": 1,
    "generalization_status_counts": dict(status_counts),
    "next_problem": (
        "Add provenance, atomic claims, authority treatment, "
        "and contradiction control."
    ),
    "permission_state": {
        "observe": True,
        "organize": True,
        "browse": True,
        "internal_strategy_analysis": True,
        "model_risk_review": True,
        "recommendation_v2": False,
        "external_action": False
    }
})

state_path.write_text(
    json.dumps(state, indent=2),
    encoding="utf-8"
)

audit_record = {
    "timestamp": datetime.datetime.now().isoformat(),
    "step": 2,
    "action": (
        "Tested generalization across five heterogeneous litigation contexts"
    ),
    "outputs": {
        "generalization_reviews": 5,
        "recommendation_v2_created": 0,
        "decision": "DEC-002"
    },
    "status_counts": dict(status_counts),
    "validation_passed": validation["passed"]
}

with (VAULT / "11_Audit" / "workflow_audit.jsonl").open(
    "a",
    encoding="utf-8"
) as f:
    f.write(json.dumps(audit_record) + "\n")

print(json.dumps(state, indent=2))
